# 第 6 讲实验：Kernel、Triton 与性能分析

本实验从 CPU 参考实现开始，再在可用的 CUDA + Triton 环境中运行自定义 kernel。当前环境若缺少 Triton 或 CUDA，会跳过 GPU kernel 测试，不把 CPU 结果伪装成 GPU 结果。

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

START = Path.cwd().resolve()
REPO_ROOT = next((p for p in (START, *START.parents) if (p / "code" / "lecture_06").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("请在课程仓库内启动 Jupyter Lab")
sys.path.insert(0, str(REPO_ROOT / "code" / "lecture_06"))
from kernel_lab import (
    TRITON_AVAILABLE, gelu_reference, softmax_reference, row_sum_reference,
    matmul_relu_reference, max_abs_error, relative_l2_error,
    gelu_triton, softmax_triton, row_sum_triton, matmul_relu_triton,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)
print("Triton installed:", TRITON_AVAILABLE)
if not (TRITON_AVAILABLE and device.type == "cuda"):
    print("GPU kernels will be skipped in this environment; CPU references remain executable.")

## 实验 1：参考实现与输出语义

先确认 reference 的数学定义。Triton kernel 必须与这里的语义一致，再谈性能。

In [ ]:
torch.manual_seed(6)
x = torch.linspace(-5, 5, 257)
gelu_y = gelu_reference(x)
matrix = torch.randn(5, 2051)
row_sum_y = row_sum_reference(matrix)
softmax_y = softmax_reference(matrix[:, :129])
a = torch.randn(37, 51)
b = torch.randn(51, 29)
matmul_y = matmul_relu_reference(a, b)
print("GELU:", tuple(gelu_y.shape))
print("row sum:", tuple(row_sum_y.shape), "| expected row length =", matrix.shape[1])
print("softmax:", tuple(softmax_y.shape), "| row-sum max error =", (softmax_y.sum(-1) - 1).abs().max().item())
print("matmul + ReLU:", tuple(matmul_y.shape))
assert torch.isfinite(gelu_y).all() and torch.isfinite(softmax_y).all()
assert torch.allclose(softmax_y.sum(-1), torch.ones(5), atol=1e-6)

## 实验 2：Softmax 数值稳定性

直接对大 logits 求 exp 可能溢出；先减去行最大值不会改变 softmax 概率，因为分子和分母同乘一个常数因子。

In [ ]:
logits = torch.tensor([[1000.0, 1001.0, 999.0], [0.0, 0.0, 100.0]])
naive_exp = torch.exp(logits)
naive = naive_exp / naive_exp.sum(dim=-1, keepdim=True)
stable = softmax_reference(logits)
print("naive exp finite:", torch.isfinite(naive_exp).all().item())
print("naive softmax:", naive)
print("stable softmax:", stable)
assert not torch.isfinite(naive).all()
assert torch.isfinite(stable).all()
assert torch.allclose(stable.sum(-1), torch.ones(2), atol=1e-6)

## 实验 3：融合的理想化数据流量账本

对 N 个 FP32 元素，忽略 bias 的缓存读取：两个独立的 elementwise 操作可能读写一次中间 tensor；融合后可以避免完整中间 tensor 的写回与重读。实际流量要用 profiler 验证。

In [ ]:
N = 1_000_000
bytes_per_element = 4
separate_ops_bytes = 4 * N * bytes_per_element  # input read + intermediate write/read + output write
fused_op_bytes = 2 * N * bytes_per_element      # input read + output write
print(f"separate elementwise ops, idealized full-tensor traffic: {separate_ops_bytes / 1e6:.1f} MB")
print(f"fused elementwise op, idealized full-tensor traffic: {fused_op_bytes / 1e6:.1f} MB")
print(f"idealized traffic ratio: {separate_ops_bytes / fused_op_bytes:.1f}x")
print("Caveat: this is a simplified model, not measured HBM traffic.")

## 实验 4：可选的 Triton / CUDA 正确性与性能检查

只有在 Triton 和 CUDA 都可用时才执行。代码先比较结果误差，再计时；编译/warm-up 与 steady-state timing 分开处理。

In [ ]:
if TRITON_AVAILABLE and device.type == "cuda":
    x_gpu = torch.randn(257, device=device, dtype=torch.float32)
    matrix_gpu = torch.randn(7, 129, device=device, dtype=torch.float32)
    row_gpu = torch.randn(5, 2051, device=device, dtype=torch.float32)
    a_gpu = torch.randn(37, 51, device=device, dtype=torch.float32)
    b_gpu = torch.randn(51, 29, device=device, dtype=torch.float32)

    benchmarks = [
        ("GELU", lambda: gelu_reference(x_gpu), lambda: gelu_triton(x_gpu)),
        ("Softmax", lambda: softmax_reference(matrix_gpu), lambda: softmax_triton(matrix_gpu)),
        ("Row sum", lambda: row_sum_reference(row_gpu), lambda: row_sum_triton(row_gpu, block=256)),
        ("Matmul+ReLU", lambda: matmul_relu_reference(a_gpu, b_gpu),
         lambda: matmul_relu_triton(a_gpu, b_gpu, 16, 16, 32)),
    ]

    def cuda_event_ms(fn, warmup=5, iterations=30):
        for _ in range(warmup):
            output = fn()  # also triggers Triton JIT compilation on first call
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(iterations):
            output = fn()
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / iterations

    for name, reference_fn, triton_fn in benchmarks:
        reference_output = reference_fn()
        triton_output = triton_fn()
        torch.cuda.synchronize()
        abs_error = max_abs_error(reference_output, triton_output)
        rel_error = relative_l2_error(reference_output, triton_output)
        reference_ms = cuda_event_ms(reference_fn)
        triton_ms = cuda_event_ms(triton_fn)
        print(
            f"{name}: max abs error={abs_error:.3e}, relative L2 error={rel_error:.3e}, "
            f"PyTorch={reference_ms:.4f} ms, Triton={triton_ms:.4f} ms, "
            f"speedup={reference_ms / triton_ms:.2f}x"
        )
else:
    print("SKIP: Triton/CUDA unavailable. No GPU kernel correctness or performance result is claimed.")

## 复盘

1. 为什么最后一个 elementwise tile 必须 mask？
2. Softmax 为什么使用 -inf 作为无效列的值，而 row sum 使用 0？
3. Matmul 的 A/B tile 分别是什么形状？K-loop 在每轮更新什么？
4. 为什么 CPU reference 测试通过不能证明 Triton JIT kernel 正确？
5. 真实 GPU benchmark 需要记录哪些配置与统计量？